# Data Mining Assignment — Data Lake Preprocessing with Apache Spark

**Name:** Sahin Kaushar  
**Roll Number:** LCB2023025

This notebook creates a simple Bronze/Silver Data Lake structure, inspects the raw `sales.csv` using PySpark, performs basic preprocessing, saves the Silver output, and verifies the result.

> The Bronze copy is kept unchanged. No Gold layer or detailed analysis is performed.


In [ ]:
# 1. Install PySpark
!pip -q install pyspark


In [ ]:
# 2. Imports and paths
import os
import shutil
from pyspark.sql import SparkSession, functions as F, types as T

BASE = "/content/data_lake"
BRONZE = f"{BASE}/bronze"
SILVER = f"{BASE}/silver/processed_sales"

os.makedirs(BRONZE, exist_ok=True)
os.makedirs(SILVER, exist_ok=True)

print("Bronze:", BRONZE)
print("Silver:", SILVER)


## 3. Load the provided dataset into the Bronze layer

The original dataset is copied without editing. If `sales.csv` is already uploaded to Colab, it is used directly. Otherwise, the cell downloads the class-provided dataset from the assignment repository.


In [ ]:
# 3. Put the raw dataset in Bronze without modification

source_file = "/content/sales.csv"
bronze_file = f"{BRONZE}/sales.csv"

if not os.path.exists(source_file):
    !wget -q -O /content/sales.csv https://raw.githubusercontent.com/necromancersanju-netizen/data-mining-data-lake-assignment/main/dataset/sales.csv

shutil.copy2(source_file, bronze_file)
print("Bronze file created:", bronze_file)


In [ ]:
# 4. Create SparkSession
spark = (
    SparkSession.builder
    .appName("DataMiningDataLakeAssignment")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)


## 5. Load and inspect the Bronze data

In [ ]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(bronze_file)
)

print("Schema:")
df.printSchema()

print("First 5 rows:")
df.show(5, truncate=False)

print("Total records:", df.count())


## 6. Data quality inspection

The checks below identify:
- exact duplicate rows
- null values
- text-format inconsistencies
- invalid quantity
- invalid unit price
- invalid discount percentage
- invalid dates


In [ ]:
# Exact duplicate rows
duplicate_count = df.count() - df.dropDuplicates().count()
print("Exact duplicate rows:", duplicate_count)

# Null counts by column
null_counts = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])
print("Null counts:")
null_counts.show()


In [ ]:
# Text-format inconsistencies: distinct category values after trimming/lowercasing
for c in ["category", "payment_method", "city", "state", "order_status"]:
    if c in df.columns:
        print(f"\n{c}:")
        df.select(c).where(F.col(c).isNotNull()).distinct().orderBy(c).show(100, truncate=False)


In [ ]:
# Invalid numeric values
numeric_checks = df.select(
    F.sum((F.col("quantity").isNull() | (F.col("quantity") <= 0)).cast("int")).alias("invalid_quantity"),
    F.sum((F.col("unit_price").isNull() | (F.col("unit_price") <= 0)).cast("int")).alias("invalid_unit_price"),
    F.sum((F.col("discount_percent").isNull() | (F.col("discount_percent") < 0) | (F.col("discount_percent") > 100)).cast("int")).alias("invalid_discount")
)
numeric_checks.show()


In [ ]:
# Date validation using Spark try_to_timestamp.
# Supports common date formats. Clearly invalid dates become NULL.

raw_date = F.trim(F.col("order_date").cast("string"))

date_check = df.withColumn(
    "parsed_order_date",
    F.coalesce(
        F.try_to_timestamp(raw_date, F.lit("yyyy-MM-dd")),
        F.try_to_timestamp(raw_date, F.lit("dd/MM/yyyy")),
        F.try_to_timestamp(raw_date, F.lit("MM/dd/yyyy"))
    ).cast("date")
)

print("Rows with invalid/unparseable dates:")
date_check.filter(
    F.col("order_date").isNotNull() & F.col("parsed_order_date").isNull()
).select("order_id", "order_date").show(50, truncate=False)

## 7. Silver preprocessing

Rules used:
1. Remove exact duplicate rows.
2. Trim whitespace in text fields.
3. Standardize obvious capitalization inconsistencies in text fields.
4. Convert numeric columns to numeric types.
5. Treat non-positive quantity and unit price as invalid and replace them with null.
6. Treat discount values outside 0–100% as invalid and replace them with null.
7. Parse valid dates from `yyyy-MM-dd`, `dd/MM/yyyy`, or `MM/dd/yyyy` into one consistent date type; clearly invalid dates become null.
8. Fill missing text values with `"Unknown"` and numeric missing values with the median of the valid Silver records.
9. Keep useful records rather than dropping rows unnecessarily.

In [ ]:
# 7A. Remove exact duplicates and trim text fields
silver = df.dropDuplicates()

text_cols = ["customer_name", "product", "category", "payment_method", "city", "state", "order_status"]
for c in text_cols:
    if c in silver.columns:
        silver = silver.withColumn(c, F.trim(F.col(c)))

# Standardize obvious capitalization inconsistencies.
silver = silver.withColumn("category", F.initcap(F.lower(F.col("category"))))
silver = silver.withColumn("payment_method", F.initcap(F.lower(F.col("payment_method"))))
silver = silver.withColumn("city", F.initcap(F.lower(F.col("city"))))
silver = silver.withColumn("state", F.initcap(F.lower(F.col("state"))))
silver = silver.withColumn("order_status", F.initcap(F.lower(F.col("order_status"))))


In [ ]:
# 7B. Numeric validation
silver = (
    silver
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("unit_price", F.col("unit_price").cast("double"))
    .withColumn("discount_percent", F.col("discount_percent").cast("double"))
)

silver = silver.withColumn(
    "quantity",
    F.when(F.col("quantity") > 0, F.col("quantity")).otherwise(F.lit(None).cast("int"))
)

silver = silver.withColumn(
    "unit_price",
    F.when(F.col("unit_price") > 0, F.col("unit_price")).otherwise(F.lit(None).cast("double"))
)

silver = silver.withColumn(
    "discount_percent",
    F.when(
        (F.col("discount_percent") >= 0) & (F.col("discount_percent") <= 100),
        F.col("discount_percent")
    ).otherwise(F.lit(None).cast("double"))
)


In [ ]:
# 7C. Date standardization
# Parse valid dates into one consistent Spark date type.
# Clearly invalid dates (for example 31/02/2026) become NULL.

raw_date = F.trim(F.col("order_date").cast("string"))

silver = silver.withColumn(
    "order_date",
    F.coalesce(
        F.try_to_timestamp(raw_date, F.lit("yyyy-MM-dd")),
        F.try_to_timestamp(raw_date, F.lit("dd/MM/yyyy")),
        F.try_to_timestamp(raw_date, F.lit("MM/dd/yyyy"))
    ).cast("date")
)

In [ ]:
# 7D. Handle missing values.
# Text: explicit Unknown label.
for c in ["customer_name", "product", "category", "payment_method", "city", "state", "order_status"]:
    silver = silver.withColumn(c, F.coalesce(F.col(c), F.lit("Unknown")))

# Numeric: median of valid values, calculated with Spark approxQuantile.
for c in ["quantity", "unit_price", "discount_percent"]:
    median = silver.approxQuantile(c, [0.5], 0.01)[0]
    silver = silver.withColumn(c, F.coalesce(F.col(c), F.lit(float(median))))

# Invalid dates are rare/clearly invalid, so retain the record but label the date as null.


In [ ]:
# 7E. Save Silver layer as a single CSV file
# Spark writes CSV output as a directory, so coalesce(1) creates one part file.
silver_output_dir = f"{SILVER}/csv_output"

(
    silver.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(silver_output_dir)
)

# Rename the single part file to the required sales_silver.csv
part_files = [
    f for f in os.listdir(silver_output_dir)
    if f.startswith("part-") and f.endswith(".csv")
]
assert len(part_files) == 1, f"Expected one CSV part, found {part_files}"

final_csv = f"{SILVER}/sales_silver.csv"
if os.path.exists(final_csv):
    os.remove(final_csv)

os.rename(
    os.path.join(silver_output_dir, part_files[0]),
    final_csv
)
shutil.rmtree(silver_output_dir)

print("Silver CSV saved to:", final_csv)
print("Silver CSV size (bytes):", os.path.getsize(final_csv))

## 8. Verification

In [ ]:
before_count = df.count()
after_count = silver.count()

print("Row count before preprocessing:", before_count)
print("Row count after preprocessing:", after_count)

print("\nSilver schema:")
silver.printSchema()

print("\nSample Silver rows:")
silver.show(10, truncate=False)


In [ ]:
# Final checks
print("Remaining exact duplicates:", after_count - silver.dropDuplicates().count())

print("Remaining null counts:")
silver.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in silver.columns
]).show()

print("Silver output exists:", os.path.exists(final_csv))
if os.path.exists(final_csv):
    print("Silver output path:", final_csv)

## 9. Preprocessing summary

- Exact duplicate rows were removed.
- Leading/trailing spaces were trimmed from text fields.
- Obvious text capitalization inconsistencies were standardized.
- Quantity and unit price values that were non-positive were treated as invalid.
- Discount percentages outside 0–100 were treated as invalid.
- Valid dates were converted to a consistent date type; clearly invalid dates were retained as null rather than deleting the whole record.
- Missing text values were represented as `Unknown`.
- Missing/invalid numeric values were filled using median values calculated from valid records.
- The processed result was saved as `sales_silver.csv` in the Silver layer.
- No Gold layer or detailed analysis was performed.
